# DQN vs DRQN on Hexxed

Trains and compares DQN and DRQN agents on the Hexxed puzzle environment.  
Hyperparameter tuning via **Optuna** · Experiment tracking via **MLflow**.

**Research question:** Which agent generalises faster across all 6 levels within a fixed timestep budget?

**Objective metric:** `Σ (total_timesteps − step_when_level_cleared)` for each cleared level — higher means faster learning.

## Setup

In [ ]:
import os
import random
from pathlib import Path
from collections import deque
from io import BytesIO

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gym import core, spaces

import mlflow
from mlflow.tracking import MlflowClient
import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import requests

## Configuration

`spawn.txt` must be in the same directory as this notebook (the project root).

In [ ]:
SPAWN_FILE = Path.cwd() / 'spawn.txt'

ENV_CONFIG = {
    'num_vertices':    6,
    'step_per_pattern': 6,
    'levels':          6,
    'shuffle_patterns': True,
    'random_rolls':    False,
    'perfect_bonus':   False,
    'render_mode':     0,
}

TRAIN_CONFIG = {
    'total_timesteps': 10000,
    'eval_episodes':   20,
    'epsilon_start':   1.0,
    'epsilon_end':     0.05,
    'epsilon_decay':   0.99,
    'log_every':       100,
    'rolling_window':  10,
}

# Optuna search spaces — (type, low, high, log_scale)
DQN_SEARCH_SPACE = {
    'lr':           ('float', 1e-4, 1e-2, True),
    'gamma':        ('float', 0.90, 0.999, False),
    'buffer_size':  ('int',   5_000, 50_000, False),
    'batch_size':   ('int',   32, 256, False),
    'target_update':('int',   10, 500, False),
    'hidden_dim':   ('int',   64, 512, False),
}

DRQN_SEARCH_SPACE = {
    'lr':           ('float', 1e-4, 1e-2, True),
    'gamma':        ('float', 0.90, 0.999, False),
    'buffer_size':  ('int',   5_000, 50_000, False),
    'batch_size':   ('int',   32, 256, False),
    'target_update':('int',   10, 500, False),
    'hidden_dim':   ('int',   64, 512, False),
    'seq_len':      ('int',   4, 12, False),  # capped — early episodes are short
}

# MLflow
MLFLOW_TRACKING_URI  = 'mlruns'
DQN_EXPERIMENT_NAME  = 'hexxed_dqn'
DRQN_EXPERIMENT_NAME = 'hexxed_drqn'

# Optuna
OPTUNA_DIR      = 'optuna_studies'
DQN_STUDY_NAME  = 'dqn_study'
DRQN_STUDY_NAME = 'drqn_study'
N_TRIALS        = 50

## Environment — Hexxed

A hexagonal puzzle game. The agent sees the full grid each step (full observability).  
Each episode is one puzzle. The agent rotates the grid and locks rows to score points.  
Max reward per locked row = 36. Levels 1–6 have 1–6 pieces respectively (max 36–216 per episode).

In [ ]:
class hexxed(core.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self):
        self.curr_wave   = 1
        self.curr_act    = 0
        self.num_attempts = 0
        self.wave_reward = 0
        self.max_reward  = 0
        self.subwave_num = 0

    def ready(self, num_vertices=6, step_per_pattern=6, levels=6,
              shuffle_patterns=True, random_rolls=True,
              normalize_reward=False, perfect_bonus=False, render_mode=1):
        self.grid_x          = num_vertices
        self.cut_off         = step_per_pattern
        self.num_levels      = levels
        self.grid_y          = 2 * num_vertices
        self.shuffle_patterns = shuffle_patterns
        self.random_rolls    = random_rolls
        self.normalize_reward = normalize_reward
        self.render_mode     = render_mode
        self.perfect_bonus   = 36 if perfect_bonus else 0
        self.pattern_list    = self.read_patterns(range(self.curr_wave))
        self.action_space    = spaces.Discrete(num_vertices + 1)
        grid_size            = self.grid_x * self.grid_y
        self.observation_space = spaces.Box(
            np.zeros(grid_size, dtype=np.float32),
            np.ones(grid_size,  dtype=np.float32),
            dtype=np.float32
        )
        self.subwave_id  = []
        self.reward_mean = []
        self.level_num   = []
        self.attempt_num = []
        self.reward_hist = []
        self.level_hist  = []
        self.action_hist = []

    def read_patterns(self, num_targets):
        all_patterns = np.loadtxt(SPAWN_FILE, int, delimiter=',')
        patterns = []
        for i in num_targets:
            level_patterns = all_patterns[all_patterns[..., 1] == i + 1, ...]
            level_patterns = np.append(
                level_patterns[..., 0].reshape((-1, i + 1)),
                level_patterns[::i + 1, 3].reshape((-1, 1)), 1
            )
            if self.shuffle_patterns:
                np.random.shuffle(level_patterns)
            patterns.extend(level_patterns.tolist())
        return patterns

    def step(self, action):
        if self.render_mode:
            self.render()
        self.curr_act = action
        self.action_hist.append(action)
        self.subwave_id.append(self.subwave_num)
        reward = 0
        done   = 0
        if action < 6:
            self.step_grid(action)
        else:
            if self.ismoving and np.sum(self.grid[0, self.cut_off:]):
                reward = (np.where(self.grid[0, self.cut_off:])[0][0] + 1) ** 2
                self.grid[0, :] = 0
            else:
                self.step_grid(0)
        self.ismoving = True
        if self.normalize_reward:
            reward /= 36.0
        done = np.sum(self.grid) == 0
        self.subwave_reward += max(0, reward)
        if done:
            if self.subwave_reward == self.subwave_max:
                self.max_reward += self.perfect_bonus * self.curr_wave
                reward += self.perfect_bonus * self.curr_wave
            self.max_reward  += self.pattern_list[self.subwave_num][-1]
            self.subwave_num  = self.subwave_num + 1
        self.wave_reward += max(0, reward)
        self.reward_hist.append(self.subwave_reward)
        self.reward_mean.append(self.wave_reward)
        self.level_hist.append(self.curr_wave)
        self.attempt_num.append(self.num_attempts)
        if self.render_mode:
            print(self.curr_act, end=' ')
            print(self.curr_wave, end=' ')
            print(self.num_attempts, end=' ')
            print(self.subwave_reward)
        return self.grid.flatten(), reward, done, {}

    def step_grid(self, vert):
        if self.ismoving:
            dist = max(1, min(vert, 6 - vert))
        else:
            dist = 1
        self.grid = np.roll(np.roll(self.grid, -vert, 0), dist, 1)
        self.grid[:, :dist] = 0

    def reset(self):
        if (self.curr_wave < 6 and self.wave_reward < self.max_reward - len(self.pattern_list) * 15) or \
           (self.curr_wave == 6 and self.wave_reward < self.max_reward - len(self.pattern_list) * 6):
            self.reset_helper()
        elif self.subwave_num == len(self.pattern_list):
            self.reset_helper()
            if self.curr_wave == self.num_levels:
                self.curr_wave = 1
            else:
                self.curr_wave = min(self.curr_wave + 1, self.num_levels)
        if self.subwave_num == 0:
            self.pattern_list = self.read_patterns(range(self.curr_wave - 1, self.curr_wave))
            self.wave_reward  = 0
            self.max_reward   = 0
            self.num_attempts += 1
        self.ismoving    = False
        self.subwave_max = self.pattern_list[self.subwave_num][-1]
        self.subwave_reward = 0
        self.grid = np.zeros(shape=(self.grid_x, self.grid_y))
        for i in range(len(self.pattern_list[self.subwave_num]) - 1):
            self.grid[self.pattern_list[self.subwave_num][i]][self.cut_off - i - 1] = 1
        if self.random_rolls:
            self.grid = np.roll(self.grid, random.randint(0, self.grid_x), 0)
            if random.randint(0, 2):
                self.grid = np.roll(np.flip(self.grid, 0), 1, 0)
        return self.grid.flatten()

    def reset_helper(self):
        self.subwave_num = 0

    def render(self, mode='human', close=False):
        if close:
            return
        hexagon = np.zeros(self.grid_x)
        print(' ', end='')
        for x in range(self.grid_x):
            y = np.where(self.grid[x, :])[0]
            if np.size(y):
                hexagon[x] = y[0]
        for x in range(self.grid_x):
            out_val = int(hexagon[x] - 5)
            if out_val == -5:
                print('.', end='  ')
            elif out_val < 0:
                print(out_val, end=' ')
            else:
                print(out_val, end='  ')

## Agent Networks

**DQN** — standard feedforward Q-network. Appropriate for fully observable environments.

**DRQN** — adds an LSTM layer to maintain hidden state across timesteps within an episode. Useful when observations are partial; in Hexxed the grid is fully visible so DRQN's memory provides no structural advantage.

In [ ]:
class DQNNetwork(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)


class DRQNNetwork(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU()
        )
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder = nn.Linear(hidden_dim, action_dim)

    def forward(self, x, hidden=None):
        encoded   = self.encoder(x)
        lstm_out, hidden = self.lstm(encoded, hidden)
        q_values  = self.decoder(lstm_out)
        return q_values, hidden

    def init_hidden(self, batch_size=1, device='cpu'):
        return (
            torch.zeros(1, batch_size, self.hidden_dim).to(device),
            torch.zeros(1, batch_size, self.hidden_dim).to(device)
        )

## Replay Buffers

**ReplayBuffer** — stores individual `(s, a, r, s', done)` transitions. Used by DQN.

**SequenceReplayBuffer** — stores complete episodes and samples fixed-length sequences from them. Used by DRQN. Only episodes longer than `seq_len` are kept. `is_ready` checks total stored transitions (not episode count) to avoid the buffer starving the training loop during short early-level episodes.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, float(done)))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(np.array(actions)),
            torch.FloatTensor(np.array(rewards)),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(np.array(dones))
        )

    def __len__(self):
        return len(self.buffer)

    def is_ready(self, batch_size):
        return len(self) >= batch_size


class SequenceReplayBuffer:
    def __init__(self, capacity, seq_len):
        self.buffer          = deque(maxlen=capacity)
        self.seq_len         = seq_len
        self.current_episode = []
        self._total_transitions = 0

    def push(self, state, action, reward, next_state, done):
        self.current_episode.append((state, action, reward, next_state, float(done)))
        if done:
            if len(self.current_episode) >= self.seq_len:
                if len(self.buffer) == self.buffer.maxlen:
                    self._total_transitions -= len(self.buffer[0])
                self._total_transitions += len(self.current_episode)
                self.buffer.append(self.current_episode)
            self.current_episode = []

    def sample(self, batch_size):
        sequences = []
        while len(sequences) < batch_size:
            episode = random.choice(self.buffer)
            start   = random.randint(0, len(episode) - self.seq_len)
            sequences.append(episode[start: start + self.seq_len])

        states, actions, rewards, next_states, dones = [], [], [], [], []
        for seq in sequences:
            s, a, r, ns, d = zip(*seq)
            states.append(s)
            actions.append(a)
            rewards.append(r)
            next_states.append(ns)
            dones.append(d)

        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(np.array(actions)),
            torch.FloatTensor(np.array(rewards)),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(np.array(dones))
        )

    def __len__(self):
        return len(self.buffer)

    def is_ready(self, batch_size):
        return self._total_transitions >= batch_size

## Progress Tracking & MLflow Logging

`ProgressTracker` records per-episode data and milestones during a training run.  
The `learning_speed_score` is the Optuna objective: `Σ (total_steps − step_at_level_clear)` for all cleared levels.

In [ ]:
class ProgressTracker:
    def __init__(self, rolling_window=10):
        self.rolling_window  = rolling_window
        self.steps           = []
        self.episode_rewards = []
        self.episode_steps   = []
        self.level_at_step   = []
        self.rolling_reward  = []
        self.first_level_clear  = None
        self.level_clear_steps  = {}
        self.episodes_to_level  = {}
        self._episode_count  = 0

    def log_episode(self, step, reward, level, env):
        self._episode_count += 1
        self.steps.append(step)
        self.episode_rewards.append(reward)
        self.episode_steps.append(step)
        self.level_at_step.append(level)

        window = self.episode_rewards[-self.rolling_window:]
        self.rolling_reward.append(np.mean(window))

        if level not in self.level_clear_steps:
            if env.wave_reward >= env.max_reward - len(env.pattern_list) * 6:
                self.level_clear_steps[level]  = step
                self.episodes_to_level[level]  = self._episode_count
                if level == 1 and self.first_level_clear is None:
                    self.first_level_clear = step
                print(f'  ★ Level {level} cleared at step {step} '
                      f'(episode {self._episode_count})')

    def learning_speed_score(self, total_steps):
        if not self.level_clear_steps:
            return 0.0
        return float(sum(total_steps - step for step in self.level_clear_steps.values()))

    def summary(self):
        return {
            'first_level_clear_step': self.first_level_clear,
            'level_clear_steps':      self.level_clear_steps,
            'episodes_to_level':      self.episodes_to_level,
            'final_rolling_reward':   self.rolling_reward[-1] if self.rolling_reward else 0,
            'mean_reward':            np.mean(self.episode_rewards) if self.episode_rewards else 0,
        }

In [ ]:
def setup_mlflow(agent_type):
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    experiment = DQN_EXPERIMENT_NAME if agent_type == 'dqn' else DRQN_EXPERIMENT_NAME
    mlflow.set_experiment(experiment)


def start_run(params):
    run = mlflow.start_run()
    mlflow.log_params(params)
    return run


def end_run():
    mlflow.end_run()


def log_episode(episode, reward, epsilon, level, loss=None):
    mlflow.log_metric('episode_reward', reward,  step=episode)
    mlflow.log_metric('epsilon',        epsilon, step=episode)
    mlflow.log_metric('level',          level,   step=episode)
    if loss is not None:
        mlflow.log_metric('loss', loss, step=episode)


def log_trial_result(score):
    mlflow.log_metric('learning_speed_score', score)


def log_summary(tracker):
    summary = tracker.summary()

    fls = summary['first_level_clear_step']
    mlflow.log_metric('first_level_clear_step', fls if fls is not None else -1)

    for level, step in summary['level_clear_steps'].items():
        mlflow.log_metric(f'level_{level}_clear_step', step)

    for level, ep in summary['episodes_to_level'].items():
        mlflow.log_metric(f'level_{level}_clear_episode', ep)

    mlflow.log_metric('final_rolling_reward', summary['final_rolling_reward'])
    mlflow.log_metric('mean_reward',          summary['mean_reward'])

    rewards = tracker.episode_rewards
    if len(rewards) >= 3:
        n = len(rewards) // 3
        mlflow.log_metric('reward_early', float(np.mean(rewards[:n])))
        mlflow.log_metric('reward_mid',   float(np.mean(rewards[n:2*n])))
        mlflow.log_metric('reward_late',  float(np.mean(rewards[2*n:])))

## Training

Both loops share the same structure:
1. Epsilon-greedy action selection
2. Push transition to replay buffer
3. Train when buffer is ready
4. Sync target network every `target_update` steps
5. Stop early if all 6 levels are cleared (prevents the environment's level-reset corrupting metrics)

Return value is the **learning speed score** — the Optuna objective.

In [ ]:
def _dqn_train_step(policy_net, target_net, optimizer,
                    buffer, batch_size, gamma, device):
    states, actions, rewards, next_states, dones = buffer.sample(batch_size)
    states      = states.to(device)
    actions     = actions.to(device)
    rewards     = rewards.to(device)
    next_states = next_states.to(device)
    dones       = dones.to(device)

    current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        max_next_q = target_net(next_states).max(dim=1)[0]
        target_q   = rewards + gamma * max_next_q * (1 - dones)

    loss = F.mse_loss(current_q, target_q)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


def train_dqn(lr, gamma, batch_size, buffer_size, target_update, hidden_dim):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    env = hexxed()
    env.ready(**ENV_CONFIG)

    nv         = ENV_CONFIG['num_vertices']
    obs_dim    = nv * 2 * nv
    action_dim = nv + 1

    policy_net = DQNNetwork(obs_dim, action_dim, hidden_dim).to(device)
    target_net = DQNNetwork(obs_dim, action_dim, hidden_dim).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = torch.optim.Adam(policy_net.parameters(), lr=lr)
    buffer    = ReplayBuffer(capacity=buffer_size)
    tracker   = ProgressTracker(rolling_window=TRAIN_CONFIG['rolling_window'])

    epsilon       = TRAIN_CONFIG['epsilon_start']
    epsilon_end   = TRAIN_CONFIG['epsilon_end']
    epsilon_decay = TRAIN_CONFIG['epsilon_decay']
    total_steps   = TRAIN_CONFIG['total_timesteps']

    step    = 0
    episode = 0

    while step < total_steps:
        state         = env.reset()
        episode_reward = 0
        last_loss     = None
        done          = False

        while not done:
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    state_t  = torch.FloatTensor(state).unsqueeze(0).to(device)
                    q_values = policy_net(state_t)
                    action   = q_values.argmax().item()

            next_state, reward, done, _ = env.step(action)
            buffer.push(state, action, reward, next_state, done)
            state          = next_state
            episode_reward += reward
            step           += 1

            if buffer.is_ready(batch_size):
                last_loss = _dqn_train_step(
                    policy_net, target_net, optimizer,
                    buffer, batch_size, gamma, device
                )

            if step % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())

            epsilon = max(epsilon_end, epsilon * epsilon_decay)

        episode += 1
        tracker.log_episode(step, episode_reward, env.curr_wave, env)
        log_episode(episode, episode_reward, epsilon, env.curr_wave, last_loss)

        if ENV_CONFIG['levels'] in tracker.level_clear_steps:
            break

    return tracker.learning_speed_score(total_steps), tracker

In [ ]:
def _drqn_train_step(policy_net, target_net, optimizer,
                     buffer, batch_size, gamma, device):
    states, actions, rewards, next_states, dones = buffer.sample(batch_size)
    states      = states.to(device)
    actions     = actions.to(device)
    rewards     = rewards.to(device)
    next_states = next_states.to(device)
    dones       = dones.to(device)

    hidden        = policy_net.init_hidden(batch_size=batch_size, device=device)
    target_hidden = target_net.init_hidden(batch_size=batch_size, device=device)

    q_all, _      = policy_net(states, hidden)
    next_q_all, _ = target_net(next_states, target_hidden)

    # burn-in: only backprop on the last step of each sequence
    q_last       = q_all[:, -1, :]
    next_q_last  = next_q_all[:, -1, :]
    actions_last = actions[:, -1]
    rewards_last = rewards[:, -1]
    dones_last   = dones[:, -1]

    current_q = q_last.gather(1, actions_last.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        max_next_q = next_q_last.max(dim=1)[0]
        target_q   = rewards_last + gamma * max_next_q * (1 - dones_last)

    loss = F.mse_loss(current_q, target_q)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=10)
    optimizer.step()
    return loss.item()


def train_drqn(lr, gamma, batch_size, buffer_size, target_update, hidden_dim, seq_len):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    env = hexxed()
    env.ready(**ENV_CONFIG)

    nv         = ENV_CONFIG['num_vertices']
    obs_dim    = nv * 2 * nv
    action_dim = nv + 1

    policy_net = DRQNNetwork(obs_dim, action_dim, hidden_dim).to(device)
    target_net = DRQNNetwork(obs_dim, action_dim, hidden_dim).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = torch.optim.Adam(policy_net.parameters(), lr=lr)
    buffer    = SequenceReplayBuffer(capacity=buffer_size, seq_len=seq_len)
    tracker   = ProgressTracker(rolling_window=TRAIN_CONFIG['rolling_window'])

    epsilon       = TRAIN_CONFIG['epsilon_start']
    epsilon_end   = TRAIN_CONFIG['epsilon_end']
    epsilon_decay = TRAIN_CONFIG['epsilon_decay']
    total_steps   = TRAIN_CONFIG['total_timesteps']

    step    = 0
    episode = 0

    while step < total_steps:
        state          = env.reset()
        done           = False
        episode_reward = 0
        last_loss      = None
        hidden         = policy_net.init_hidden(batch_size=1, device=device)

        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0).unsqueeze(0).to(device)

            if random.random() < epsilon:
                action = env.action_space.sample()
                with torch.no_grad():
                    _, hidden = policy_net(state_t, hidden)
            else:
                with torch.no_grad():
                    q_values, hidden = policy_net(state_t, hidden)
                    action = q_values[0, 0, :].argmax().item()

            next_state, reward, done, _ = env.step(action)
            buffer.push(state, action, reward, next_state, done)
            state          = next_state
            episode_reward += reward
            step           += 1

            if buffer.is_ready(batch_size):
                last_loss = _drqn_train_step(
                    policy_net, target_net, optimizer,
                    buffer, batch_size, gamma, device
                )

            if step % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())

            epsilon = max(epsilon_end, epsilon * epsilon_decay)

        episode += 1
        tracker.log_episode(step, episode_reward, env.curr_wave, env)
        log_episode(episode, episode_reward, epsilon, env.curr_wave, last_loss)

        if ENV_CONFIG['levels'] in tracker.level_clear_steps:
            break

    return tracker.learning_speed_score(total_steps), tracker

## Hyperparameter Search — Optuna

`build_params` translates the search space config into Optuna suggestions.  
Each objective function wraps a full training run as a single Optuna trial.  
`run_study` creates or resumes a persistent SQLite-backed study — safe to interrupt and restart.

In [ ]:
def build_params(trial, search_space):
    params = {}
    for name, (kind, low, high, log) in search_space.items():
        if kind == 'float':
            params[name] = trial.suggest_float(name, low, high, log=log)
        elif kind == 'int':
            params[name] = trial.suggest_int(name, low, high)
    return params


def make_dqn_objective():
    def objective(trial):
        params = build_params(trial, DQN_SEARCH_SPACE)
        setup_mlflow('dqn')
        start_run(params)
        try:
            score, tracker = train_dqn(**params)
            log_trial_result(score)
            log_summary(tracker)
            return score
        except Exception as e:
            import traceback
            traceback.print_exc()
            return float('-inf')
        finally:
            end_run()
    return objective


def make_drqn_objective():
    def objective(trial):
        params = build_params(trial, DRQN_SEARCH_SPACE)
        setup_mlflow('drqn')
        start_run(params)
        try:
            score, tracker = train_drqn(**params)
            log_trial_result(score)
            log_summary(tracker)
            return score
        except Exception as e:
            import traceback
            traceback.print_exc()
            return float('-inf')
        finally:
            end_run()
    return objective


def run_study(agent_type):
    os.makedirs(OPTUNA_DIR, exist_ok=True)

    if agent_type == 'dqn':
        study_name = DQN_STUDY_NAME
        objective  = make_dqn_objective()
    else:
        study_name = DRQN_STUDY_NAME
        objective  = make_drqn_objective()

    db_path = f'sqlite:///{OPTUNA_DIR}/{study_name}.db'
    study   = optuna.create_study(
        study_name=study_name,
        storage=db_path,
        direction='maximize',
        load_if_exists=True,
        sampler=TPESampler(seed=42)
    )

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    print(f'Starting {agent_type.upper()} study — {N_TRIALS} trials')
    if study.trials:
        print(f'Resuming from trial {len(study.trials)}')

    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f'\nBest {agent_type.upper()} params (score: {study.best_value:.0f}):')
    for k, v in study.best_params.items():
        print(f'  {k}: {v}')

    return study.best_params

## Run

Set `AGENT` to `'dqn'` or `'drqn'` and run the cell. Results are saved to `mlruns/` and `optuna_studies/`.  
The study is resumable — interrupt and re-run the cell to continue from where it left off.

In [ ]:
AGENT = 'dqn'  # 'dqn' or 'drqn'

best_params = run_study(AGENT)

print(f'\nBest params found for {AGENT.upper()}:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

## Analysis & Visualisation

Loads completed runs from MLflow and produces a 2×2 comparison figure:
1. Learning curves (best trial each agent)
2. Reward by training phase (all trials)
3. Distribution of first level-clear step (all trials)
4. Level progression over episodes (mean ± std, all trials)

Run both agents first, then execute the cells below.

In [ ]:
def get_client():
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    return MlflowClient()


def get_runs(client, experiment_name):
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        raise ValueError(f'Experiment {experiment_name!r} not found in mlruns/')
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="status = 'FINISHED'",
        order_by=['metrics.learning_speed_score DESC']
    )
    print(f'{experiment_name}: {len(runs)} completed runs')
    return runs


def get_metric_curve(client, run_id, metric_name):
    history = client.get_metric_history(run_id, metric_name)
    if not history:
        return [], []
    return [m.step for m in history], [m.value for m in history]


def get_scalar(run, metric_name, default=None):
    return run.data.metrics.get(metric_name, default)


def get_all_scalars(runs, metric_name):
    return [r.data.metrics[metric_name] for r in runs if metric_name in r.data.metrics]


def smooth(values, window=10):
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode='valid')


def load_human_data():
    url = 'https://drive.google.com/uc?export=download&id=1m4RPxkOraYFfZLHQ7yhGdM6Y3oeTP8YP'
    r   = requests.get(url, allow_redirects=True)
    df  = pd.read_parquet(BytesIO(r.content))
    winners = df[(df['learning'] == True) & (df['learned'] == True)]
    print(f'Human data loaded: {len(winners)} winner rows')
    return winners


def print_summary_table(dqn_runs, drqn_runs):
    metrics = [
        ('learning_speed_score',   'Learning speed score'),
        ('first_level_clear_step', 'Steps to clear L1'),
        ('reward_early',           'Reward (early)'),
        ('reward_mid',             'Reward (mid)'),
        ('reward_late',            'Reward (late)'),
        ('final_rolling_reward',   'Final rolling reward'),
        ('mean_reward',            'Mean reward'),
    ]
    print('\n' + '=' * 65)
    print(f'{"Metric":<30} {"DQN":>15} {"DRQN":>15}')
    print('=' * 65)
    for key, label in metrics:
        dqn_vals  = [v for v in get_all_scalars(dqn_runs,  key) if v >= 0]
        drqn_vals = [v for v in get_all_scalars(drqn_runs, key) if v >= 0]
        if not dqn_vals and not drqn_vals:
            continue
        dqn_str  = f'{np.mean(dqn_vals):.1f} ± {np.std(dqn_vals):.1f}'  if dqn_vals  else 'n/a'
        drqn_str = f'{np.mean(drqn_vals):.1f} ± {np.std(drqn_vals):.1f}' if drqn_vals else 'n/a'
        if dqn_vals and drqn_vals:
            lower_is_better = 'step' in key
            winner = '<- DQN' if (
                np.mean(dqn_vals) < np.mean(drqn_vals) if lower_is_better
                else np.mean(dqn_vals) > np.mean(drqn_vals)
            ) else 'DRQN ->'
        else:
            winner = ''
        print(f'{label:<30} {dqn_str:>15} {drqn_str:>15}  {winner}')
    print('=' * 65)

In [ ]:
def plot_learning_curves(client, dqn_runs, drqn_runs, ax):
    for runs, label, color in [
        (dqn_runs,  'DQN',  '#DD8452'),
        (drqn_runs, 'DRQN', '#55A868')
    ]:
        run = runs[0] if runs else None
        if run is None:
            continue
        _, values = get_metric_curve(client, run.info.run_id, 'episode_reward')
        if not values:
            continue
        smoothed = smooth(values, window=10)
        ax.plot(range(len(smoothed)), smoothed, label=label, color=color, linewidth=2)
        clear_ep = get_scalar(run, 'level_1_clear_episode')
        if clear_ep is not None:
            ax.axvline(clear_ep, color=color, linestyle='--', alpha=0.6,
                       label=f'{label} clears L1 (ep {int(clear_ep)})')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward (smoothed)')
    ax.set_title('1 — Learning curves (best trial each)')
    ax.legend(fontsize=8)


def plot_phase_breakdown(dqn_runs, drqn_runs, ax):
    phases = ['early', 'mid', 'late']
    labels = ['Early\n(first third)', 'Mid\n(second third)', 'Late\n(final third)']
    dqn_means  = [np.mean(get_all_scalars(dqn_runs,  f'reward_{p}')) for p in phases]
    drqn_means = [np.mean(get_all_scalars(drqn_runs, f'reward_{p}')) for p in phases]
    dqn_stds   = [np.std(get_all_scalars(dqn_runs,   f'reward_{p}')) for p in phases]
    drqn_stds  = [np.std(get_all_scalars(drqn_runs,  f'reward_{p}')) for p in phases]
    x = np.arange(3)
    w = 0.35
    ax.bar(x - w/2, dqn_means,  w, yerr=dqn_stds,  label='DQN',  color='#DD8452', alpha=0.85, capsize=4)
    ax.bar(x + w/2, drqn_means, w, yerr=drqn_stds, label='DRQN', color='#55A868', alpha=0.85, capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Average reward')
    ax.set_title('2 — Reward by training phase (all trials)')
    ax.legend(fontsize=8)


def plot_first_clear_distribution(dqn_runs, drqn_runs, ax):
    dqn_clears  = [v for v in get_all_scalars(dqn_runs,  'first_level_clear_step') if v > 0]
    drqn_clears = [v for v in get_all_scalars(drqn_runs, 'first_level_clear_step') if v > 0]
    if not dqn_clears and not drqn_clears:
        ax.text(0.5, 0.5, 'No level clears recorded yet',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title('3 — First level clear distribution')
        return
    all_vals = dqn_clears + drqn_clears
    bins = np.linspace(min(all_vals), max(all_vals), 20)
    ax.hist(dqn_clears,  bins=bins, alpha=0.6, label=f'DQN (n={len(dqn_clears)})',
            color='#DD8452', density=True)
    ax.hist(drqn_clears, bins=bins, alpha=0.6, label=f'DRQN (n={len(drqn_clears)})',
            color='#55A868', density=True)
    if dqn_clears:
        ax.axvline(np.mean(dqn_clears),  color='#DD8452', linestyle='--', linewidth=1.5,
                   label=f'DQN mean: {np.mean(dqn_clears):.0f}')
    if drqn_clears:
        ax.axvline(np.mean(drqn_clears), color='#55A868', linestyle='--', linewidth=1.5,
                   label=f'DRQN mean: {np.mean(drqn_clears):.0f}')
    ax.set_xlabel('Training steps to clear level 1')
    ax.set_ylabel('Density')
    ax.set_title('3 — First level clear distribution (all trials)')
    ax.legend(fontsize=8)


def plot_level_progression(client, dqn_runs, drqn_runs, ax):
    def avg_level_curve(runs):
        all_curves = []
        for run in runs:
            _, values = get_metric_curve(client, run.info.run_id, 'level')
            if values:
                all_curves.append(values)
        if not all_curves:
            return [], []
        max_len = max(len(c) for c in all_curves)
        padded  = [c + [c[-1]] * (max_len - len(c)) for c in all_curves]
        return np.mean(padded, axis=0), np.std(padded, axis=0)

    for runs, label, color in [
        (dqn_runs,  'DQN',  '#DD8452'),
        (drqn_runs, 'DRQN', '#55A868')
    ]:
        mean, std = avg_level_curve(runs)
        if len(mean):
            eps = range(len(mean))
            ax.plot(eps, mean, label=label, color=color, linewidth=2)
            ax.fill_between(eps, mean - std, mean + std, color=color, alpha=0.15)

    ax.set_xlabel('Episode')
    ax.set_ylabel('Level')
    ax.set_title('4 — Level progression (mean ± std, all trials)')
    ax.set_yticks(range(1, 7))
    ax.legend(fontsize=8)


def run_analysis(save=True):
    client    = get_client()
    dqn_runs  = get_runs(client, DQN_EXPERIMENT_NAME)
    drqn_runs = get_runs(client, DRQN_EXPERIMENT_NAME)

    if not dqn_runs:
        print('No DQN runs found — run the DQN study first.')
        return
    if not drqn_runs:
        print('No DRQN runs found — run the DRQN study first.')
        return

    print_summary_table(dqn_runs, drqn_runs)

    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    plot_learning_curves(client, dqn_runs, drqn_runs, ax1)
    plot_phase_breakdown(dqn_runs, drqn_runs, ax2)
    plot_first_clear_distribution(dqn_runs, drqn_runs, ax3)
    plot_level_progression(client, dqn_runs, drqn_runs, ax4)

    fig.suptitle(
        f'DQN vs DRQN — Learning speed on Hexxed ({TRAIN_CONFIG["total_timesteps"]:,} steps)',
        fontsize=14, y=1.01
    )

    if save:
        os.makedirs('analysis', exist_ok=True)
        plt.savefig('analysis/comparison.png', dpi=150, bbox_inches='tight')
        print('Saved to analysis/comparison.png')

    plt.show()

In [ ]:
run_analysis(save=True)